# Lançador Kaggle — does-ontology-matter

**Só edite a célula de configuração (abaixo).** Escolha o fluxo com `MODO`:

- `MODO = "inducao_c2"` — o modelo induz a ontologia C2 das 29 ontologias e gera as
  definições D-24 (`configs/c2/v_full/<slug>/`). Faz push verificado.
- `MODO = "experimento"` — roda o `MANIFESTO` indicado (ex.: `manifesto_7b_completo.json`),
  com guard check da C2 e push por experimento.

Rode todas as células na ordem (Run All). Nenhuma outra célula precisa de edição manual.
Para acelerar a instalação, anexe o Dataset do wheelhouse (ver `docs/kaggle_wheelhouse.md`).

In [ ]:
# ===================== CONFIGURAÇÃO — EDITE AQUI =====================
MODO = "inducao_c2"   # "inducao_c2"  ou  "experimento"
MANIFESTO = "configs/pipeline/manifesto_7b_completo.json"   # usado no modo experimento
MODELO = "Qwen/Qwen2.5-7B-Instruct-AWQ"
# =====================================================================

import os, json
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
GITHUB_TOKEN = secrets.get_secret('GITHUB_TOKEN')
HF_TOKEN     = secrets.get_secret('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

REPO_SLUG    = 'felipemoraissoares/does-ontology-matter'
REPO_URL     = f'https://{GITHUB_TOKEN}@github.com/{REPO_SLUG}.git'
BENCH_URL    = 'https://github.com/cenguix/Text2KGBench.git'
BENCH_COMMIT = '50a3d255371b8817cdff70fd88459ac82b339cfe'

WORK_DIR  = '/kaggle/working'
REPO_DIR  = f'{WORK_DIR}/does-ontology-matter'
BENCH_DIR = f'{WORK_DIR}/Text2KGBench'
VLLM_LOG  = f'{WORK_DIR}/vllm.log'
PORT      = 8000
GIT_EMAIL = 'felipemoraissoares@gmail.com'
GIT_NAME  = 'Felipe Morais (Kaggle)'

assert MODO in ('inducao_c2', 'experimento'), f'MODO invalido: {MODO}'
print('MODO     =', MODO)
if MODO == 'experimento':
    print('MANIFESTO=', MANIFESTO)
print('MODELO   =', MODELO)

In [ ]:
# Helpers
import subprocess

def run(cmd, check=True, **kw):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if result.stdout.strip(): print(result.stdout.strip())
    if result.stderr.strip(): print(result.stderr.strip())
    if check and result.returncode != 0:
        raise RuntimeError(f'Falhou (rc={result.returncode}): {cmd[:120]}')
    return result

def git(cmd):
    return run(f'git -C "{REPO_DIR}" {cmd}', check=False)

In [ ]:
# Clonar repositórios
if not os.path.exists(REPO_DIR):
    run(f'git clone --depth=1 {REPO_URL} {REPO_DIR}')
else:
    print('Repo ja existe - atualizando...'); git('pull')

if not os.path.exists(BENCH_DIR):
    run(f'git init {BENCH_DIR}')
    run(f'git -C {BENCH_DIR} fetch --depth=1 {BENCH_URL} {BENCH_COMMIT}')
    run(f'git -C {BENCH_DIR} checkout FETCH_HEAD')
    actual = run(f'git -C {BENCH_DIR} rev-parse HEAD', check=False).stdout.strip()
    print('Text2KGBench OK' if actual == BENCH_COMMIT else f'AVISO bench commit {actual[:12]}')
else:
    print('Text2KGBench ja existe.')

git(f'config user.email "{GIT_EMAIL}"')
git(f'config user.name "{GIT_NAME}"')
print('Clones OK.')

In [ ]:
# Instalar dependencias (wheelhouse se anexado; senao, requirements travado D-35)
import glob

REQ = f'{REPO_DIR}/requirements-kaggle.txt'
wheelhouse = None
for _d in glob.glob('/kaggle/input/*'):
    if glob.glob(os.path.join(_d, '*.whl')):
        wheelhouse = _d; break

if wheelhouse:
    print(f'Wheelhouse detectado: {wheelhouse} - instalando offline.')
    run(f'pip install -q --no-index --find-links {wheelhouse} -r {REQ}')
else:
    print('Wheelhouse nao anexado - instalando do PyPI com versoes travadas.')
    run(f'pip install -q -r {REQ}')

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
import sys; sys.path.insert(0, f'{REPO_DIR}/src')
print('Dependencias OK.')

In [ ]:
# Subir vLLM (dtype=half obrigatorio na T4; tensor-parallel-size=2 usa as 2 GPUs)
vllm_cmd = (
    f'vllm serve {MODELO} --port {PORT} --dtype half '
    f'--tensor-parallel-size 2 --max-model-len 4096 '
    f'--gpu-memory-utilization 0.90 > {VLLM_LOG} 2>&1 &'
)
os.system(vllm_cmd)
print(f'vLLM iniciado. Log: {VLLM_LOG}')

In [ ]:
# Aguardar vLLM pronto
import time, urllib.request, urllib.error
TIMEOUT, INTERVALO, elapsed = 360, 10, 0
while elapsed < TIMEOUT:
    try:
        urllib.request.urlopen(f'http://localhost:{PORT}/health', timeout=3)
        print(f'vLLM pronto apos {elapsed}s.'); break
    except (urllib.error.URLError, ConnectionRefusedError, OSError):
        if elapsed % 30 == 0: print(f'  aguardando vLLM... ({elapsed}s/{TIMEOUT}s)')
        time.sleep(INTERVALO); elapsed += INTERVALO
else:
    run(f'tail -40 {VLLM_LOG}', check=False)
    raise RuntimeError('vLLM nao iniciou.')

In [ ]:
# Executar o fluxo escolhido (rodar_kaggle.py despacha conforme MODO)
PYPATH = {**os.environ, 'PYTHONPATH': f'{REPO_DIR}/src'}
base = (f'python {REPO_DIR}/scripts/rodar_kaggle.py --modo {MODO} '
        f'--repo_dir {REPO_DIR} --bench_dir {BENCH_DIR} --work_dir {WORK_DIR} '
        f'--modelo {MODELO} --url http://localhost:{PORT}/v1')
if MODO == 'experimento':
    base += f' --manifesto {REPO_DIR}/{MANIFESTO}'
run(base, cwd=REPO_DIR, env=PYPATH)
print(f'\nConcluido: MODO={MODO}')

In [ ]:
# Diagnostico: ultimas linhas do log do vLLM
run(f'tail -30 {VLLM_LOG}', check=False)